# Treinamento no Google Colab — CNN (PyTorch) para Imagens Intraorais

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taynaramos/dental-image-classifier/blob/main/notebooks/pytorch_kfold_colab.ipynb)

Versão do pipeline de treino preparada para rodar no **Google Colab com GPU**, reaproveitando o código de `src/pytorch_kfold` — o comportamento é idêntico ao da CLI (`kfold-train`) e ao notebook local (`pytorch_kfold_training.ipynb`).

**Antes de executar:**
1. `Ambiente de execução → Alterar tipo de ambiente de execução → GPU` (no Colab Pro, prefira **A100** ou **L4**).
2. O dataset é baixado automaticamente na seção 2 — nenhuma configuração necessária.

Todas as dependências do projeto (PyTorch com CUDA, torchvision, scikit-learn, matplotlib) já vêm pré-instaladas no Colab — **não** rode `pip install -r requirements-pytorch-kfold.txt` aqui, pois as versões pinadas substituiriam o PyTorch com suporte a GPU do Colab.

## 1. Clonar o repositório

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/taynaramos/dental-image-classifier.git"
    BRANCH = "pytorch-kfold"  # troque para "main" após o merge
    PROJECT_ROOT = Path("/content/dental-image-classifier")
    if not PROJECT_ROOT.exists():
        !git clone --branch {BRANCH} {REPO_URL} {PROJECT_ROOT}
else:
    # Permite executar este mesmo notebook localmente, a partir de experiments/
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Projeto em: {PROJECT_ROOT}")

## 2. Download do dataset

O zip do dataset é baixado direto do Google Drive via `gdown` (link compartilhado "qualquer pessoa com o link") — **sem precisar montar o Drive nem autorizar acesso**. O arquivo é salvo no disco local da VM e extraído em `data/dataset`.

In [ ]:
import shutil
import zipfile

# ID do arquivo no Google Drive (link "qualquer pessoa com o link pode ver")
GDRIVE_FILE_ID = "11XFNZe6KpP2Jkhb6SkAOuYkl3rfEoOX9"

DATASET_ROOT = PROJECT_ROOT / "data" / "dataset"

if IN_COLAB and not DATASET_ROOT.exists():
    zip_local = Path("/content/Dataset_CINUFPE_Odontologico.zip")
    if not zip_local.exists():
        !gdown {GDRIVE_FILE_ID} -O {zip_local}

    print("Extraindo …")
    with zipfile.ZipFile(zip_local) as zf:
        zf.extractall(DATASET_ROOT)
    # Se o zip tiver uma única pasta raiz, desaninha para DATASET_ROOT
    conteudos = list(DATASET_ROOT.iterdir())
    if len(conteudos) == 1 and conteudos[0].is_dir():
        raiz_unica = conteudos[0]
        for filho in raiz_unica.iterdir():
            shutil.move(str(filho), DATASET_ROOT)
        raiz_unica.rmdir()

n_sujeitos = sum(1 for p in DATASET_ROOT.iterdir() if p.is_dir())
print(f"Dataset em : {DATASET_ROOT}")
print(f"Sujeitos   : {n_sujeitos}")

## 3. Importações e verificação da GPU

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.pytorch_kfold.dataset import build_dataloaders, resolve_imagefolder_root
from src.pytorch_kfold.model import DentalCNN, ModelConfig
from src.pytorch_kfold.predict import predict
from src.pytorch_kfold.trainer import Trainer
from src.pytorch_kfold.utils import get_device, save_checkpoint, set_seed

%matplotlib inline

device = get_device()
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sem GPU — selecione um ambiente de execução com GPU para acelerar o treino.")

## 4. Configuração

Mesmos hiperparâmetros do notebook local; `BATCH_SIZE` e `NUM_WORKERS` maiores para aproveitar a GPU.

In [ ]:
MODEL_OUT = PROJECT_ROOT / "artifacts" / "kfold_model.pth"

IMAGE_SIZE    = 128
GRAYSCALE     = True
BATCH_SIZE    = 64
NUM_WORKERS   = 2
EPOCHS        = 20
PATIENCE      = 5      # early stopping: nº de épocas sem melhora na val loss antes de parar
MIN_DELTA     = 0.0    # melhora mínima na val loss para zerar a paciência
LEARNING_RATE = 1e-3
SEED          = 42

set_seed(SEED)
print(f"Modelo (out): {MODEL_OUT}")

## 5. Preparo do dataset e DataLoaders

In [ ]:
# Materializa (uma única vez) o dataset por sujeito em train/val/test por classe.
# A divisão é feita por sujeito, para não vazar dados do mesmo paciente entre conjuntos.
imagefolder_root = resolve_imagefolder_root(DATASET_ROOT, seed=SEED)

train_loader, val_loader, test_loader, classes = build_dataloaders(
    imagefolder_root,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    grayscale=GRAYSCALE,
    num_workers=NUM_WORKERS,
)
NUM_CLASSES = len(classes)

print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Batches — treino: {len(train_loader)}  val: {len(val_loader)}  teste: {len(test_loader)}")

## 6. Visualização de algumas imagens

In [ ]:
classes_preview = sorted(p.name for p in (imagefolder_root / "train").iterdir() if p.is_dir())

fig, axes = plt.subplots(1, len(classes_preview), figsize=(15, 3))
for ax, classe in zip(axes, classes_preview):
    exemplo = random.choice(list((imagefolder_root / "train" / classe).glob("*")))
    ax.imshow(Image.open(exemplo))
    ax.set_title(classe, fontsize=9)
    ax.axis('off')
fig.suptitle("Um exemplo aleatório por classe (conjunto de treino)", y=1.02)
plt.tight_layout()
plt.show()

## 7. Modelo e treinamento

In [ ]:
config = ModelConfig(image_size=IMAGE_SIZE, grayscale=GRAYSCALE)
model = DentalCNN(num_classes=NUM_CLASSES, in_channels=config.in_channels)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {total_params:,}")

trainer = Trainer(model, device, learning_rate=LEARNING_RATE)
history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA)

## 8. Curvas de Loss e Acurácia

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.train_loss, label="Treino")
ax1.plot(history.val_loss, label="Validação")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.set_title("Curva de Loss")
ax1.legend()

ax2.plot(history.train_accuracy, label="Treino")
ax2.plot(history.val_accuracy, label="Validação")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia")
ax2.set_title("Curva de Acurácia")
ax2.legend()

plt.tight_layout()
plt.show()

## 9. Avaliação final e matriz de confusão

In [ ]:
test_loss, test_acc = trainer.evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

print(classification_report(y_true, y_pred, target_names=classes))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, cmap="Blues", xticks_rotation=45, ax=ax,
)
ax.set_title(f"Matriz de Confusão — Teste (acurácia = {test_acc:.4f})")
plt.tight_layout()
plt.show()

## 10. Predição em uma imagem de exemplo

In [ ]:
classe_exemplo = random.choice(classes)
exemplo_path = random.choice(list((imagefolder_root / "test" / classe_exemplo).glob("*")))
pred = predict(exemplo_path, model, classes, config, device)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(exemplo_path))
plt.title(f"Previsto: {pred.label}")
plt.axis('off')
plt.show()

print(f"Imagem          : {exemplo_path.name}")
print(f"Classe real     : {classe_exemplo}")
print(f"Classe prevista : {pred.label}")
print("Probabilidades  :")
for cls, prob in sorted(pred.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<20} {prob:.1%}")

## 11. Salvar o modelo

A VM do Colab é descartada ao fim da sessão. Para não perder o checkpoint, **baixe-o pelo painel de arquivos** (ícone de pasta na barra lateral → `dental-image-classifier/artifacts/kfold_model.pth` → ⋮ → Fazer download). Se o Drive estiver montado em `/content/drive`, a célula também copia o checkpoint para lá automaticamente.

In [ ]:
save_checkpoint(MODEL_OUT, model, classes, config, history=history)
print(f"Modelo salvo em: {MODEL_OUT}")

# Copia para o Drive apenas se ele estiver de fato montado (drive.mount) —
# sem o mount, /content/drive seria só uma pasta local perdida ao fim da sessão.
drive_mount = Path("/content/drive/MyDrive")
if IN_COLAB and drive_mount.is_dir():
    DRIVE_OUT = drive_mount / "dental-image-classifier" / "artifacts"
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODEL_OUT, DRIVE_OUT / MODEL_OUT.name)
    print(f"Cópia no Drive : {DRIVE_OUT / MODEL_OUT.name}")
elif IN_COLAB:
    print("Drive não montado — baixe o checkpoint pelo painel de arquivos para não perdê-lo.")

## 12. Validação cruzada k-fold (por sujeito)

Para ter uma estimativa mais confiável do desempenho (o teste de um split único tem só 45 sujeitos), fazemos validação cruzada com **k = 5 folds divididos por sujeito**: os 300 sujeitos são embaralhados e cortados em 5 grupos de 60; em cada rodada um grupo é o teste e os outros quatro (240 sujeitos) são o treino. Um **modelo novo é treinado do zero em cada fold** — nenhum peso é reaproveitado entre rodadas.

Aqui usamos o esquema clássico **treino/teste** (sem conjunto de validação): o treino roda por um **número fixo de épocas** (12, escolhido a partir do experimento da seção 7, onde o early stopping parou na época 12 sem sinal de overfitting). Assim o fold de teste não influencia nenhuma decisão de treinamento e é avaliado uma única vez ao final da rodada.

O split é feito à mão (embaralhar e fatiar a lista), sem `sklearn.model_selection`, e o loop de treino é escrito explicitamente na célula. O resultado final é a **média ± desvio padrão** da acurácia de teste nos 5 folds.

> A validação cruzada serve para **avaliar** a robustez do pipeline; o modelo entregável continua sendo o do treino da seção 7.

In [ ]:
from torch.utils.data import Dataset

# Classes na mesma ordem alfabética que o ImageFolder usa (para manter os índices iguais)
CLASSES_CV = ["frontal", "inferior", "lateral_direita", "lateral_esquerda", "superior"]

# nome do arquivo (sem extensão) -> classe
STEM_PARA_CLASSE = {
    "intraoral-frontal": "frontal",
    "intraoral-inferior": "inferior",
    "intraoral-superior": "superior",
    "intraoral-lateral-direita": "lateral_direita",
    "intraoral-lateral-esquerda": "lateral_esquerda",
}


class DatasetIntraoral(Dataset):
    """Dataset caseiro: recebe uma lista de pastas de sujeitos e monta os pares (imagem, rótulo)."""

    def __init__(self, sujeitos, transform):
        self.itens = []
        for sujeito in sujeitos:
            for caminho in sorted(sujeito.glob("*.jpeg")):
                classe = STEM_PARA_CLASSE.get(caminho.stem)
                if classe is not None:
                    self.itens.append((caminho, CLASSES_CV.index(classe)))
        self.transform = transform

    def __len__(self):
        return len(self.itens)

    def __getitem__(self, idx):
        caminho, rotulo = self.itens[idx]
        imagem = Image.open(caminho).convert("RGB")
        return self.transform(imagem), rotulo


print("DatasetIntraoral pronto")

In [ ]:
from src.pytorch_kfold.dataset import build_transform
from torch import nn
from torch.utils.data import DataLoader

K = 5          # número de folds
EPOCHS_CV = 12  # nº FIXO de épocas por fold — escolhido a partir da seção 7,
                # onde o early stopping parou na época 12 (sem overfitting até lá)

# 1) embaralha os sujeitos (semente fixa para ser reprodutível)
sujeitos = sorted(p for p in DATASET_ROOT.iterdir() if p.is_dir())
random.seed(SEED)
random.shuffle(sujeitos)

# 2) corta a lista em K pedaços (o último fica com a sobra da divisão)
tamanho_fold = len(sujeitos) // K
folds = []
for i in range(K):
    inicio = i * tamanho_fold
    fim = inicio + tamanho_fold if i < K - 1 else len(sujeitos)
    folds.append(sujeitos[inicio:fim])

print(f"{len(sujeitos)} sujeitos divididos em {K} folds de tamanhos {[len(f) for f in folds]}")

transform = build_transform(IMAGE_SIZE, GRAYSCALE)
acuracias_teste = []

# 3) uma rodada por fold: o fold k é o teste, os outros quatro são o treino
for k in range(K):
    print(f"\n========== Fold {k + 1}/{K} ==========")
    sujeitos_teste = folds[k]
    sujeitos_treino = []
    for i in range(K):
        if i != k:
            sujeitos_treino = sujeitos_treino + folds[i]
    print(f"sujeitos — treino: {len(sujeitos_treino)}  teste: {len(sujeitos_teste)}")

    loader_treino = DataLoader(
        DatasetIntraoral(sujeitos_treino, transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    )
    loader_teste = DataLoader(
        DatasetIntraoral(sujeitos_teste, transform),
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    )

    # modelo novo (do zero!) a cada fold — nada de reaproveitar pesos entre folds
    set_seed(SEED + k)
    modelo_cv = DentalCNN(num_classes=len(CLASSES_CV), in_channels=config.in_channels).to(device)
    criterio = nn.CrossEntropyLoss()
    otimizador = torch.optim.Adam(modelo_cv.parameters(), lr=LEARNING_RATE)

    # loop de treino escrito na mão, com número FIXO de épocas —
    # o fold de teste não influencia nenhuma decisão do treinamento
    for epoca in range(1, EPOCHS_CV + 1):
        modelo_cv.train()
        loss_soma, acertos, total = 0.0, 0, 0
        for imagens, rotulos in loader_treino:
            imagens, rotulos = imagens.to(device), rotulos.to(device)

            otimizador.zero_grad()
            saidas = modelo_cv(imagens)
            loss = criterio(saidas, rotulos)
            loss.backward()
            otimizador.step()

            loss_soma += loss.item() * rotulos.size(0)
            acertos += (saidas.argmax(dim=1) == rotulos).sum().item()
            total += rotulos.size(0)
        print(f"época {epoca:2d}/{EPOCHS_CV} — train loss {loss_soma / total:.4f} | train acc {acertos / total:.4f}")

    # 4) avaliação ÚNICA no fold de teste, depois do treino completo
    modelo_cv.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for imagens, rotulos in loader_teste:
            imagens, rotulos = imagens.to(device), rotulos.to(device)
            saidas = modelo_cv(imagens)
            acertos += (saidas.argmax(dim=1) == rotulos).sum().item()
            total += rotulos.size(0)
    acc_teste = acertos / total
    print(f"\n>>> Fold {k + 1}: Test Accuracy {acc_teste:.4f}")
    acuracias_teste.append(acc_teste)

In [ ]:
# Média e desvio padrão calculados "na mão" (desvio amostral, divide por K - 1)
media = sum(acuracias_teste) / K
desvio = (sum((a - media) ** 2 for a in acuracias_teste) / (K - 1)) ** 0.5

print("Acurácia de teste por fold:")
for k in range(K):
    print(f"  Fold {k + 1}: {acuracias_teste[k]:.4f}")

print(f"\nMédia : {media:.4f}")
print(f"Desvio: {desvio:.4f}")
print(f"\nResultado final: {media * 100:.1f}% ± {desvio * 100:.1f}%  (validação cruzada {K}-fold por sujeito)")